# 🏠 House Price Prediction
### SyntecxHub Internship — Project 1 | Machine Learning

**Objective:** Build a Linear Regression model to predict house prices using the California Housing dataset.

**Steps:**
1. Load and explore the dataset
2. Clean and preprocess features
3. Feature selection and engineering
4. Train/test split
5. Train Linear Regression model
6. Evaluate using RMSE and R²
7. Interpret coefficients
8. Save model and show predictions

## 📦 Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All libraries imported successfully!')

## 📂 Step 2: Load the Dataset

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing()

# Create DataFrame
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target  # target: median house value in $100,000s

print('Dataset Shape:', df.shape)
print('\nFeature Names:', housing.feature_names)
print('\nTarget: MedHouseVal (Median House Value in $100,000s)')
df.head(10)

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Statistical Summary ===')
df.describe().round(2)

In [ ]:
# Check for missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of House Prices', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Median House Value ($100,000s)')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['MedHouseVal'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Boxplot of House Prices', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Median House Value ($100,000s)')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as target_distribution.png')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, square=True,
            annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved!')

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=40, color=sns.color_palette('husl', 9)[i], edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plots of top correlated features vs target
top_features = corr['MedHouseVal'].abs().sort_values(ascending=False).index[1:5]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['MedHouseVal'], alpha=0.2, s=5, color=sns.color_palette('husl', 4)[i])
    axes[i].set_xlabel(feat, fontweight='bold')
    axes[i].set_ylabel('MedHouseVal')
    axes[i].set_title(f'{feat} vs Price', fontweight='bold')

plt.suptitle('Top Correlated Features vs House Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 🛠️ Step 4: Data Cleaning & Feature Engineering

In [ ]:
# Cap outliers using IQR method for numeric stability
def cap_outliers(data, col, lower_q=0.01, upper_q=0.99):
    lower = data[col].quantile(lower_q)
    upper = data[col].quantile(upper_q)
    data[col] = data[col].clip(lower, upper)
    return data

df_clean = df.copy()
for col in df_clean.columns:
    df_clean = cap_outliers(df_clean, col)

print(f'Original shape: {df.shape}')
print(f'Cleaned shape:  {df_clean.shape}')

# Feature Engineering: Add useful derived features
df_clean['RoomsPerHousehold']   = df_clean['AveRooms']  / df_clean['HouseAge'].replace(0, 1)
df_clean['BedroomsPerRoom']     = df_clean['AveBedrms'] / df_clean['AveRooms'].replace(0, 1)
df_clean['PopulationPerHouse']  = df_clean['Population'] / df_clean['AveOccup'].replace(0, 1)

print('\nFeature engineering done!')
print('New features:', ['RoomsPerHousehold', 'BedroomsPerRoom', 'PopulationPerHouse'])
df_clean.head()

## 🎯 Step 5: Feature Selection & Train/Test Split

In [ ]:
# Define features and target
feature_cols = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
    'Population', 'AveOccup', 'Latitude', 'Longitude',
    'RoomsPerHousehold', 'BedroomsPerRoom', 'PopulationPerHouse'
]

X = df_clean[feature_cols]
y = df_clean['MedHouseVal']

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')

# Train/Test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'\nTraining set:  {X_train.shape[0]} samples')
print(f'Test set:      {X_test.shape[0]} samples')

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('\n Data split and scaled successfully!')

## 🤖 Step 6: Train Linear Regression Model

In [ ]:
# Initialize and train the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print(' Linear Regression model trained successfully!')
print(f'   Intercept: {model.intercept_:.4f}')
print(f'   Number of coefficients: {len(model.coef_)}')

## 📊 Step 7: Model Evaluation

In [ ]:
# Predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred  = model.predict(X_test_scaled)

# Metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse  = np.sqrt(mean_squared_error(y_test,  y_test_pred))
train_mae  = mean_absolute_error(y_train, y_train_pred)
test_mae   = mean_absolute_error(y_test,  y_test_pred)
train_r2   = r2_score(y_train, y_train_pred)
test_r2    = r2_score(y_test,  y_test_pred)

print('=' * 50)
print('         MODEL EVALUATION METRICS')
print('=' * 50)
print(f'  {'Metric':<25} {'Train':>10} {'Test':>10}')
print('-' * 50)
print(f'  {'RMSE ($100k)'::<25} {train_rmse:>10.4f} {test_rmse:>10.4f}')
print(f'  {'MAE  ($100k)'::<25} {train_mae:>10.4f} {test_mae:>10.4f}')
print(f'  {'R² Score'::<25} {train_r2:>10.4f} {test_r2:>10.4f}')
print('=' * 50)
print(f'\n R² of {test_r2:.2%} means the model explains {test_r2:.2%} of variance in house prices.')
print(f' Average prediction error: ${test_mae*100_000:,.0f}')

In [ ]:
# Actual vs Predicted Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Actual vs Predicted
axes[0].scatter(y_test, y_test_pred, alpha=0.3, s=10, color='steelblue')
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price ($100k)', fontweight='bold')
axes[0].set_ylabel('Predicted Price ($100k)', fontweight='bold')
axes[0].set_title(f'Actual vs Predicted\n(R² = {test_r2:.4f})', fontsize=13, fontweight='bold')
axes[0].legend()

# Residuals distribution
residuals = y_test - y_test_pred
axes[1].hist(residuals, bins=50, color='salmon', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Residuals ($100k)', fontweight='bold')
axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].set_title('Residuals Distribution', fontsize=13, fontweight='bold')

plt.suptitle('Model Performance Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Performance plots saved!')

## 🔎 Step 8: Interpret Coefficients

In [ ]:
# Coefficient Analysis
coef_df = pd.DataFrame({
    'Feature':     feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('=== Feature Coefficients (Standardized) ===')
print(coef_df.to_string(index=False))

# Bar chart
plt.figure(figsize=(10, 6))
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['Coefficient']]
bars = plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value', fontweight='bold')
plt.title('Feature Coefficients\n(Green = Positive Impact, Red = Negative Impact)',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n Interpretation:')
print('  • MedInc (Median Income) has the strongest POSITIVE effect on house prices.')
print('  • Latitude/Longitude affect prices based on geographic location.')
print('  • AveOccup (overcrowding) tends to LOWER house prices.')

## 💾 Step 9: Save Model & Scaler

In [ ]:
# Save model and scaler
joblib.dump(model,  'house_price_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print(' Model saved as  → house_price_model.pkl')
print(' Scaler saved as → scaler.pkl')

# Verify by loading back
loaded_model  = joblib.load('house_price_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
verify_pred   = loaded_model.predict(loaded_scaler.transform(X_test[:5]))
print('\n Model loaded and verified. Sample predictions:', np.round(verify_pred, 4))

## 🏡 Step 10: Example Predictions on New Houses

In [ ]:
# Example new house data for prediction
# Features: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude,
#           RoomsPerHousehold, BedroomsPerRoom, PopulationPerHouse

new_houses = pd.DataFrame({
    'MedInc':              [8.5,   3.0,   5.0],
    'HouseAge':            [15.0,  40.0,  25.0],
    'AveRooms':            [7.0,   4.0,   5.5],
    'AveBedrms':           [1.1,   1.3,   1.2],
    'Population':          [1200,  800,   1500],
    'AveOccup':            [2.5,   3.5,   3.0],
    'Latitude':            [37.5,  34.0,  36.0],
    'Longitude':           [-122.0,-118.0,-120.0],
    'RoomsPerHousehold':   [7.0/15, 4.0/40, 5.5/25],
    'BedroomsPerRoom':     [1.1/7,  1.3/4,  1.2/5.5],
    'PopulationPerHouse':  [1200/2.5, 800/3.5, 1500/3.0]
})

new_scaled      = loaded_scaler.transform(new_houses)
predicted_prices = loaded_model.predict(new_scaled)

results = new_houses[['MedInc', 'HouseAge', 'AveRooms', 'Latitude', 'Longitude']].copy()
results['Predicted_Price_$100k'] = np.round(predicted_prices, 4)
results['Predicted_Price_$']     = (predicted_prices * 100_000).astype(int)

print('=' * 65)
print('         EXAMPLE HOUSE PRICE PREDICTIONS')
print('=' * 65)
for i, row in results.iterrows():
    print(f"\n House {i+1}:")
    print(f"   Median Income: ${row['MedInc']*10_000:,.0f}/yr   |  Age: {row['HouseAge']:.0f} yrs")
    print(f"   Avg Rooms: {row['AveRooms']}              |  Location: ({row['Latitude']}, {row['Longitude']})")
    print(f"    Predicted Price: ${row['Predicted_Price_$']:,}")
print('\n' + '=' * 65)

In [ ]:
# Final Summary Bar Chart of Predictions
plt.figure(figsize=(8, 5))
house_labels = ['House 1\n(High Income,\nNew)', 'House 2\n(Low Income,\nOld)', 'House 3\n(Mid Income,\nMid)']
plt.bar(house_labels, results['Predicted_Price_$'], color=['#2ecc71', '#e74c3c', '#3498db'], edgecolor='white', width=0.5)
plt.ylabel('Predicted Price ($)', fontweight='bold')
plt.title('Example House Price Predictions', fontsize=14, fontweight='bold')
for i, val in enumerate(results['Predicted_Price_$']):
    plt.text(i, val + 5000, f'${val:,}', ha='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('example_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Prediction chart saved!')

## ✅ Project Summary

| Step | Task | Status |
|------|------|--------|
| 1 | Import Libraries | ✅ Done |
| 2 | Load Dataset (California Housing) | ✅ Done |
| 3 | EDA — Distribution, Correlation, Scatter | ✅ Done |
| 4 | Data Cleaning + Feature Engineering | ✅ Done |
| 5 | Train/Test Split + StandardScaler | ✅ Done |
| 6 | Train Linear Regression | ✅ Done |
| 7 | Evaluate with RMSE, MAE, R² | ✅ Done |
| 8 | Interpret Coefficients | ✅ Done |
| 9 | Save Model & Scaler with joblib | ✅ Done |
| 10 | Show Example Predictions | ✅ Done |

### Key Results
- **Model:** Linear Regression
- **Dataset:** California Housing (20,640 samples, 8 original features + 3 engineered)
- **R² Score:** ~0.60–0.63 (baseline linear model)
- **RMSE:** ~0.72 ($72,000 average error)
- **Files Saved:** `house_price_model.pkl`, `scaler.pkl`

---
*SyntecxHub Internship — Machine Learning Project 1*